In [ ]:
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from typing import TypedDict

from app.core.config import settings

from langchain.chat_models import init_chat_model

# 定初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={"thinking":{"type":"disabled"}}
)

# 定义状态
class SimpleAgentState(TypedDict):
    user_input:str
    result:str

def call_llm(state: SimpleAgentState):
    ai_message = model.invoke(state["user_input"])
    return {"result":ai_message.text}

# 3.定义边，构建图
builder = StateGraph(SimpleAgentState)

builder.add_node("llm", call_llm)

builder.add_edge(START, "llm")
builder.add_edge("llm", END)

graph = builder.compile()

# 4.调用Graph
result = graph.invoke({
    "user_input":"你是什么模型?"
})
print(f"最终结果:\n{result}")